# Working

**Request:** [T05] Billing vs ledger for the month

> New email from Sai Suram <sai@agents.agentstore.it.com>
> Subject: [T05] Billing vs ledger for the month
> Thread ID: AAQkADI1N2Y5MTE3LTE1MDctNGY0Yy1iYzQ5LWEzNmE5NzAyYzk4NQAQAOXyawY8v6lBmz4S-vSknoo=
> 
> ## T

Each cell below is one run in the sandbox, in the order it happened, with whatever it printed and produced.


In [1]:
import pandas as pd
import io

billing_data = """Invoice,Amount
INV-001,1200.00
INV-002,850.00
INV-003,2300.00
INV-004,640.00
INV-005,1750.00"""

ledger_data = """Invoice,Amount
INV-001,1200.00
INV-002,850.00
INV-003,2030.00
INV-005,1750.00
INV-005,1750.00"""

df_billing = pd.read_csv(io.StringIO(billing_data))
df_ledger = pd.read_csv(io.StringIO(ledger_data))

# Group by Invoice and sum Amount for ledger to handle duplicates
df_ledger_grouped = df_ledger.groupby('Invoice')['Amount'].sum().reset_index()

# Merge dataframes to find discrepancies
merged_df = pd.merge(df_billing, df_ledger_grouped, on='Invoice', how='outer', suffixes=('_billing', '_ledger'))

# Fill NaN with 0 for calculation
merged_df = merged_df.fillna(0)

# Calculate difference
merged_df['Difference'] = merged_df['Amount_billing'] - merged_df['Amount_ledger']

# Identify differences
differences = merged_df[merged_df['Difference'] != 0]

# Identify invoices in billing but not ledger
billing_only = df_billing[~df_billing['Invoice'].isin(df_ledger['Invoice'])]

# Identify invoices in ledger but not billing (after grouping, this means new invoices, not duplicates)
ledger_only = df_ledger_grouped[~df_ledger_grouped['Invoice'].isin(df_billing['Invoice'])]

# Identify duplicate entries in ledger (original ledger data)
duplicate_ledger_entries = df_ledger[df_ledger.duplicated(subset=['Invoice'], keep=False)]


print("Differences in Amount:")
print(differences)
print("\nInvoices in Billing but not in Ledger:")
print(billing_only)
print("\nInvoices in Ledger but not in Billing (after grouping):")
print(ledger_only)
print("\nDuplicate entries in Ledger:")
print(duplicate_ledger_entries)

# Prepare data for Excel output
with pd.ExcelWriter('/tmp/output/billing_ledger_reconciliation.xlsx', engine='xlsxwriter') as writer:
    df_billing.to_excel(writer, sheet_name='Billing Data', index=False)
    df_ledger.to_excel(writer, sheet_name='Ledger Data', index=False)
    merged_df.to_excel(writer, sheet_name='Reconciliation', index=False)

    # Add a summary sheet
    summary_df = pd.DataFrame({
        'Metric': ['Total Billing Amount', 'Total Ledger Amount', 'Total Gap (Billing - Ledger)'],
        'Value': [df_billing['Amount'].sum(), df_ledger['Amount'].sum(), df_billing['Amount'].sum() - df_ledger['Amount'].sum()]
    })
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

    if not differences.empty:
        differences.to_excel(writer, sheet_name='Amount Differences', index=False)
    else:
        pd.DataFrame([{'Status': 'No differences in amounts found.'}]).to_excel(writer, sheet_name='Amount Differences', index=False)

    if not billing_only.empty:
        billing_only.to_excel(writer, sheet_name='Billing Only Invoices', index=False)
    else:
        pd.DataFrame([{'Status': 'No invoices in Billing only.'}]).to_excel(writer, sheet_name='Billing Only Invoices', index=False)

    if not ledger_only.empty:
        ledger_only.to_excel(writer, sheet_name='Ledger Only Invoices', index=False)
    else:
        pd.DataFrame([{'Status': 'No invoices in Ledger only.'}]).to_excel(writer, sheet_name='Ledger Only Invoices', index=False)

    if not duplicate_ledger_entries.empty:
        duplicate_ledger_entries.to_excel(writer, sheet_name='Duplicate Ledger Entries', index=False)
    else:
        pd.DataFrame([{'Status': 'No duplicate entries in Ledger.'}]).to_excel(writer, sheet_name='Duplicate Ledger Entries', index=False)

print("Excel workbook created successfully at /tmp/output/billing_ledger_reconciliation.xlsx")

Differences in Amount:
   Invoice  Amount_billing  Amount_ledger  Difference
2  INV-003          2300.0         2030.0       270.0
3  INV-004           640.0            0.0       640.0
4  INV-005          1750.0         3500.0     -1750.0

Invoices in Billing but not in Ledger:
   Invoice  Amount
3  INV-004   640.0

Invoices in Ledger but not in Billing (after grouping):
Empty DataFrame
Columns: [Invoice, Amount]
Index: []

Duplicate entries in Ledger:
   Invoice  Amount
3  INV-005  1750.0
4  INV-005  1750.0
Excel workbook created successfully at /tmp/output/billing_ledger_reconciliation.xlsx



[files written: billing_ledger_reconciliation.xlsx]
